In [ ]:
!pip -q install condacolab

import condacolab
condacolab.install()

⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:09
🔁 Restarting kernel...


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/gpmhc/gpmhc_train

Mounted at /content/drive
/content/drive/MyDrive/gpmhc/gpmhc_train
/content/drive/MyDrive/gpmhc/gpmhc_train
total 1.4G
-rw------- 1 root root  438 Jun 16 02:45 environment.yml
drwx------ 2 root root 4.0K Jul 27 19:40 gpmhc
-rw------- 1 root root 3.7K Jun 16 02:45 infer.py
-rw------- 1 root root    0 Jun 16 02:45 __init__.py
-rw------- 1 root root 1.9K Jul 24 21:16 json_input.json
-rw------- 1 root root 480K Jun 16 02:45 mhc2_small_df.csv
-rw------- 1 root root 3.7M Jun 16 02:45 mhc_seq_df.csv
drwx------ 2 root root 4.0K Jul 27 19:40 models
-rw------- 1 root root 1.4G Jun 16 17:22 Presentation_df_w_preds.csv
-rw------- 1 root root 2.6K Jul 25 20:33 train.py
./environment.yml
./gpmhc/baseline_model.py
./gpmhc/data.py
./gpmhc/gnn_parts.py
./gpmhc/__init__.py
./gpmhc/learner.py
./infer.py
./__init__.py
./json_input.json
./mhc2_small_df.csv
./mhc_seq_df.csv
./Presentation_df_w_preds.csv
./train.py


In [ ]:
%%bash

cd /content/drive/MyDrive/gpmhc/gpmhc_train
conda env list | grep gpmhc || mamba env create -f environment.yml

gpmhc                  /usr/local/envs/gpmhc


In [ ]:
%%bash

cd /content/drive/MyDrive/gpmhc/gpmhc_train

conda run -n gpmhc python -c "
import torch
import numpy
import pandas
import dgl
import gpmhc

print('numpy:', numpy.__version__)
print('pandas:', pandas.__version__)
print('torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('DGL:', dgl.__version__)
print('gpmhc import OK')
"

Setting the default backend to "pytorch". You can change it in the ~/.dgl/config.json file or export the DGLBACKEND environment variable.  Valid options are: pytorch, mxnet, tensorflow (all lowercase)
numpy: 1.24.3
pandas: 2.0.2
torch: 1.13.1.post200
CUDA: True
DGL: 1.1.0+cu118
gpmhc import OK



DGL backend not selected or invalid.  Assuming PyTorch for now.



In [ ]:
%%bash
echo "Checking repo..."

python - <<'PY'
import gpmhc
from gpmhc import data
from gpmhc import gnn_parts
from gpmhc import learner

print("All imports OK")
PY

echo
echo "GPU status:"
nvidia-smi

Checking repo...

GPU status:
Mon Jul 27 21:07:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------

Traceback (most recent call last):
  File "<stdin>", line 2, in <module>
  File "/content/drive/MyDrive/gpmhc/gpmhc_train/gpmhc/data.py", line 1, in <module>
    import numpy as np
ModuleNotFoundError: No module named 'numpy'


In [ ]:
%%bash

cd /content/drive/MyDrive/gpmhc/gpmhc_train

export MPLBACKEND=Agg

echo "Starting model test..."

conda run -n gpmhc python -c "
import json
print('Python started')

from gpmhc.learner import json_to_learner
print('Imported learner')

with open('models/baseline_model/json_input.json') as f:
    config=json.load(f)

print('Loaded config')

learner=json_to_learner(config)

print('Model initialization OK')
print('Parameters:', sum(p.numel() for p in learner.model.parameters()))
"

echo "Finished"

Starting model test...
Python started
Imported learner
Loaded config
Model initialization OK
Parameters: 492101

Finished


/usr/local/envs/gpmhc/lib/python3.10/site-packages/accelerate/utils/torch_xla.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/usr/local/envs/gpmhc/lib/python3.10/site-packages/dgl/heterograph.py:92: DGLWarning: Recommend creating graphs by `dgl.graph(data)` instead of `dgl.DGLGraph(data)`.
  dgl_warning(



In [ ]:
%%bash

cd /content/drive/MyDrive/gpmhc/gpmhc_train
export MPLBACKEND=Agg
conda run -n gpmhc python -c "
import json
import pandas as pd
print('Python started')
from gpmhc.learner import json_to_learner
from gpmhc.data import cleanup_schema, tokenize, dataset
print('Imports successful')

with open('models/baseline_model/json_input.json') as f:
    config=json.load(f)
print('Config loaded')

learner=json_to_learner(config)
print('Model initialized')

df=pd.read_csv('Presentation_df_w_preds.csv', nrows=1000)
print('Raw dataframe:', df.shape)

df=cleanup_schema(
    df,
    config['dataloader_options']['csv_to_df']['schema_options']
)
print('Clean dataframe:', df.shape)

x=tokenize(df, learner.arch.tokenizer)
y=df['EL'].values.astype(int)
print('Token shape:', x.shape)
print('Label shape:', y.shape)
ds=dataset(x,y)
print('Dataset OK')
print('First sample:', ds[0][0].shape, ds[0][1])
"

Python started
Imports successful
Config loaded
Model initialized
Raw dataframe: (1000, 42)
Clean dataframe: (1000, 42)
Token shape: (1000, 728)
Label shape: (1000,)
Dataset OK
First sample: torch.Size([728]) tensor(1)



/usr/local/envs/gpmhc/lib/python3.10/site-packages/accelerate/utils/torch_xla.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/usr/local/envs/gpmhc/lib/python3.10/site-packages/dgl/heterograph.py:92: DGLWarning: Recommend creating graphs by `dgl.graph(data)` instead of `dgl.DGLGraph(data)`.
  dgl_warning(



In [ ]:
%%bash

cd /content/drive/MyDrive/gpmhc/gpmhc_train

export MPLBACKEND=Agg

cat > batch_check.py <<'PY'

import json
import torch
from torch.utils.data import DataLoader
import pandas as pd

print("Starting batch check")
from gpmhc.learner import json_to_learner
from gpmhc.data import cleanup_schema, tokenize, dataset
print("Imports done")

with open("models/baseline_model/json_input.json") as f:
    config=json.load(f)
print("Config loaded")


learner=json_to_learner(config)
print("Model loaded")
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Moving model to:", device)
learner.model = learner.model.to(device)
learner.model.eval()
print(
    "Model parameter device:",
    next(learner.model.parameters()).device
)

df=pd.read_csv(
    "Presentation_df_w_preds.csv",
    nrows=1000
)
print("Raw dataframe:", df.shape)

schema_options=config["dataloader_options"]["csv_to_df"]["schema_options"]
df=cleanup_schema(
    df,
    schema_options
)
print("Clean dataframe:", df.shape)

x=tokenize(
    df,
    learner.arch.tokenizer
)
y=df["EL"].values.astype(int)
print("Token shape:", x.shape)
print("Labels:", y.shape)
ds=dataset(x,y)
dl=DataLoader(
    ds,
    batch_size=16,
    shuffle=False
)
print("DataLoader created")

xb,yb=next(iter(dl))
print("Batch CPU:", xb.shape)
# Move batch
xb=xb.to(device)
yb=yb.to(device)
print("Batch device:", xb.device)

with torch.no_grad():
    pred=learner.model(xb)
print("Prediction shape:", pred.shape)
print("Prediction device:", pred.device)
print("SUCCESS")
PY

conda run -n gpmhc python batch_check.py

Starting batch check
Imports done
Config loaded
Model loaded
Moving model to: cuda
Model parameter device: cuda:0
Raw dataframe: (1000, 42)
Clean dataframe: (1000, 42)
Token shape: (1000, 728)
Labels: (1000,)
DataLoader created
Batch CPU: torch.Size([16, 728])
Batch device: cuda:0
Prediction shape: torch.Size([16, 32])
Prediction device: cuda:0
SUCCESS



/usr/local/envs/gpmhc/lib/python3.10/site-packages/accelerate/utils/torch_xla.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/usr/local/envs/gpmhc/lib/python3.10/site-packages/dgl/heterograph.py:92: DGLWarning: Recommend creating graphs by `dgl.graph(data)` instead of `dgl.DGLGraph(data)`.
  dgl_warning(



In [ ]:
%%bash

cd /content/drive/MyDrive/gpmhc/gpmhc_train
export MPLBACKEND=Agg

cat > dq_only_check.py <<'PY'
import json
import pandas as pd
import torch
import sys
def log(x):
    print(x, flush=True)
log("Starting DQ-only dataset check")

from gpmhc.learner import json_to_learner
from gpmhc.data import cleanup_schema
log("Imports successful")

with open("models/baseline_model/json_input.json") as f:
    config=json.load(f)
log("Config loaded")

learner=json_to_learner(config)
log("Model initialized")

df=pd.read_csv(
    "Presentation_df_w_preds.csv",
    nrows=10000
)
log(f"Raw dataframe: {df.shape}")
log("Columns:")
log(str(df.columns.tolist()))
log("Example allotypes:")
log(str(df["allotype"].head(20).tolist()))

def is_dq_only(x):
    if pd.isna(x):
        return False
    x=str(x).upper()
    # Must contain DQ
    has_dq = (
        "DQA" in x or
        "DQB" in x or
        "DQ" in x
    )
    if not has_dq:
        return False
    # Exclude DR
    if "DR" in x:
        return False
    # Exclude DP
    if "DP" in x:
        return False
    return True


dq_mask=df["allotype"].apply(is_dq_only)
dq_df=df[dq_mask].copy()
log(f"DQ-only samples: {dq_df.shape}")

log("DQ-only allotypes:")
log(str(dq_df["allotype"].value_counts()))
bad=dq_df[
    dq_df["allotype"].astype(str).str.contains(
        "DR|DP",
        case=False,
        regex=True
    )
]
log(f"Bad mixed DQ samples remaining: {bad.shape[0]}")
if bad.shape[0] > 0:
    log(str(bad["allotype"].head()))

    raise RuntimeError(
        "DQ-only filter failed. Mixed allotypes remain."
    )
log("DQ FILTER PASSED")

schema_options=config[
    "dataloader_options"
]["csv_to_df"]["schema_options"]

dq_clean=cleanup_schema(
    dq_df,
    schema_options
)
log(f"After cleanup: {dq_clean.shape}")
log("Final allotype check:")
log(str(dq_clean["allotype"].value_counts()))
log("SUCCESS: dataset is DQ-only")
PY

conda run -n gpmhc python dq_only_check.py

Starting DQ-only dataset check
Imports successful
Config loaded
Model initialized
Raw dataframe: (10000, 42)
Columns:
['Unnamed: 0.1', 'Unnamed: 0', 'peptide', 'nFlank', 'cFlank', 'allotype', 'molecule', 'data_type', 'EL', 'ic50', 'ic50_scaled', 'is_multiallotypic', 'mhc_dq1_1', 'mhc_dq1_2', 'mhc_dq1_3', 'mhc_dq1_4', 'mhc_dp1_1', 'mhc_dp1_2', 'mhc_dp1_3', 'mhc_dp1_4', 'mhc_dr1_1', 'mhc_dr1_2', 'mhc_dr3_1', 'mhc_dr3_2', 'mhc_dr4_1', 'mhc_dr4_2', 'mhc_dr5_1', 'mhc_dr5_2', 'gene_ids', 'transcript_ids', 'protein_ids', 'split', 'analysis_id', 'peptide_length', 'concat', 'cluster_label', 'Graph-pMHC_score', 'peptide_core', 'MixMHCIIPred-1.2_rank', 'mm2p_core', 'NetMHCIIPan-4.0', 'MHCNuggets_rank']
Example allotypes:
['DPA1*01:03___DPB1*03:01', 'DQA1*05:05___DQB1*03:01', 'DQA1*05:05___DQB1*03:01', 'DQA1*05:05___DQB1*03:01', 'DQA1*05:05___DQB1*03:01', 'DQA1*05:05___DQB1*03:01', 'DQA1*05:05___DQB1*03:01', 'DQA1*05:05___DQB1*03:01', 'DQA1*05:05___DQB1*03:01', 'DQA1*05:05___DQB1*03:01', 'DQA1*05:

/usr/local/envs/gpmhc/lib/python3.10/site-packages/accelerate/utils/torch_xla.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/usr/local/envs/gpmhc/lib/python3.10/site-packages/dgl/heterograph.py:92: DGLWarning: Recommend creating graphs by `dgl.graph(data)` instead of `dgl.DGLGraph(data)`.
  dgl_warning(



In [ ]:
%%bash

cd /content/drive/MyDrive/gpmhc/gpmhc_train

export MPLBACKEND=Agg
cat > dq_split_check.py <<'PY'
import pandas as pd
def log(x):
    print(x, flush=True)
log("Starting DQ-only split check")

df = pd.read_csv(
    "Presentation_df_w_preds.csv"
)

log(f"Full dataframe: {df.shape}")

def is_dq_only(allotype):

    if pd.isna(allotype):
        return False

    a = str(allotype).upper()

    # Must have DQ alpha + beta
    if "DQA" not in a:
        return False

    if "DQB" not in a:
        return False

    # Exclude DP
    if "DPA" in a or "DPB" in a:
        return False

    # Exclude DR
    if "DRB" in a or "DRA" in a or "DR" in a:
        return False
    return True

dq_df = df[df["allotype"].apply(is_dq_only)].copy()
log(f"DQ-only dataframe: {dq_df.shape}")

bad = dq_df[
    dq_df["allotype"].apply(lambda x: not is_dq_only(x))
]


log(f"Bad DQ samples remaining: {len(bad)}")
assert len(bad) == 0, "DQ contamination detected"

log("\nAvailable splits:")
log(str(dq_df["split"].value_counts()))


assert set(dq_df["split"].unique()).issubset(
    {"train","test"}
), "Unexpected split detected"


train_df = dq_df[
    dq_df["split"] == "train"
].copy()


test_df = dq_df[
    dq_df["split"] == "test"
].copy()


log("\nTRAIN")
log(str(train_df.shape))
log("\nTEST")
log(str(test_df.shape))
log("\nTRAIN allotypes")
log(str(train_df["allotype"].value_counts()))
log("\nTEST allotypes")
log(str(test_df["allotype"].value_counts()))

log("\nTRAIN EL")
log(str(train_df["EL"].value_counts(normalize=True)))
log("\nTEST EL")
log(str(test_df["EL"].value_counts(normalize=True)))
log("\nSUCCESS: DQ-only train/test subsets ready")
PY

conda run -n gpmhc python dq_split_check.py

Starting DQ-only split check
Full dataframe: (1403497, 42)
DQ-only dataframe: (46317, 42)
Bad DQ samples remaining: 0

Available splits:
split
train    26454
test     19863
Name: count, dtype: int64

TRAIN
(26454, 42)

TEST
(19863, 42)

TRAIN allotypes
allotype
DQA1*05:05___DQB1*03:01    8762
DQA1*02:01___DQB1*02:02    8468
DQA1*05:01___DQB1*02:01    4210
DQA1*01:02___DQB1*06:04    3592
DQA1*01:02___DQB1*06:02    1350
DQA1*03:01___DQB1*03:02      25
DQA1*01:01___DQB1*05:01      25
DQA1*05:01___DQB1*03:02      11
DQA1*03:01___DQB1*02:01       9
DQA1*01:02___DQB1*05:01       2
Name: count, dtype: int64

TEST allotypes
allotype
DQA1*05:05___DQB1*03:01    6245
DQA1*02:01___DQB1*02:02    5730
DQA1*05:01___DQB1*02:01    4259
DQA1*01:02___DQB1*06:04    2282
DQA1*01:02___DQB1*06:02    1008
DQA1*01:01___DQB1*05:01     124
DQA1*05:01___DQB1*03:02     117
DQA1*03:01___DQB1*02:01      71
DQA1*03:01___DQB1*03:02      27
Name: count, dtype: int64

TRAIN EL
EL
1    0.51667
0    0.48333
Name: proporti

/content/drive/MyDrive/gpmhc/gpmhc_train/dq_split_check.py:16: DtypeWarning: Columns (39) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(



In [ ]:
%%bash

cd /content/drive/MyDrive/gpmhc/gpmhc_train

cat > dq_leakage_check.py <<'PY'
import pandas as pd

def log(x):
    print(x, flush=True)
log("Starting DQ leakage check")

df = pd.read_csv(
    "Presentation_df_w_preds.csv",
    low_memory=False
)

log(f"Loaded dataframe: {df.shape}")

def is_dq_only(allotype):

    if pd.isna(allotype):
        return False

    a = str(allotype).upper()

    if "DQA" not in a:
        return False

    if "DQB" not in a:
        return False

    if "DPA" in a or "DPB" in a:
        return False

    if "DRA" in a or "DRB" in a or "DR" in a:
        return False

    return True


dq_df = df[
    df["allotype"].apply(is_dq_only)
].copy()


train_df = dq_df[
    dq_df["split"] == "train"
].copy()


test_df = dq_df[
    dq_df["split"] == "test"
].copy()


log(f"DQ train: {train_df.shape}")
log(f"DQ test: {test_df.shape}")

train_peptides = set(train_df["peptide"])
test_peptides = set(test_df["peptide"])

overlap = train_peptides.intersection(
    test_peptides
)


log("\nPeptide overlap")
log("----------------")
log(f"Unique train peptides: {len(train_peptides)}")
log(f"Unique test peptides: {len(test_peptides)}")
log(f"Shared peptides: {len(overlap)}")
log(f"Test peptide overlap fraction: {len(overlap)/len(test_peptides):.4f}")
train_allotypes = set(train_df["allotype"])
test_allotypes = set(test_df["allotype"])

log("\nAllotype overlap")
log("----------------")
log("Train-only allotypes:")
log(str(train_allotypes - test_allotypes))
log("\nTest-only allotypes:")
log(str(test_allotypes - train_allotypes))
assert len(train_df) > 0
assert len(test_df) > 0

log("\nSUCCESS: train/test separation confirmed")

PY


conda run -n gpmhc python dq_leakage_check.py

Starting DQ leakage check
Loaded dataframe: (1403497, 42)
DQ train: (26454, 42)
DQ test: (19863, 42)

Peptide overlap
----------------
Unique train peptides: 22698
Unique test peptides: 19013
Shared peptides: 0
Test peptide overlap fraction: 0.0000

Allotype overlap
----------------
Train-only allotypes:
{'DQA1*01:02___DQB1*05:01'}

Test-only allotypes:
set()

SUCCESS: train/test separation confirmed



In [ ]:
%%bash

cd /content/drive/MyDrive/gpmhc/gpmhc_train

cat > save_dq_splits.py <<'PY'

import pandas as pd
import os
def log(x):
    print(x, flush=True)
log("Saving DQ-only train/test splits")

df = pd.read_csv(
    "Presentation_df_w_preds.csv",
    low_memory=False
)

log(f"Loaded dataframe: {df.shape}")

def is_dq_only(allotype):

    if pd.isna(allotype):
        return False

    a = str(allotype).upper()

    # Require DQ alpha and beta
    if "DQA" not in a:
        return False

    if "DQB" not in a:
        return False

    # Remove DP contamination
    if "DPA" in a or "DPB" in a:
        return False

    # Remove DR contamination
    if "DRA" in a or "DRB" in a or "DR" in a:
        return False

    return True


dq_df = df[
    df["allotype"].apply(is_dq_only)
].copy()
log(f"DQ-only dataframe: {dq_df.shape}")

train_df = dq_df[
    dq_df["split"] == "train"
].copy()


test_df = dq_df[
    dq_df["split"] == "test"
].copy()

log(f"Train: {train_df.shape}")
log(f"Test: {test_df.shape}")
train_peptides = set(train_df["peptide"])
test_peptides = set(test_df["peptide"])

overlap = train_peptides.intersection(
    test_peptides
)


assert len(overlap) == 0, (
    f"Peptide leakage detected: {len(overlap)} shared peptides"
)


assert set(train_df["split"]) == {"train"}
assert set(test_df["split"]) == {"test"}

outdir = "dq_dataset"
os.makedirs(
    outdir,
    exist_ok=True
)
train_path = os.path.join(
    outdir,
    "DQ_train.csv"
)
test_path = os.path.join(
    outdir,
    "DQ_test.csv"
)
train_df.to_csv(
    train_path,
    index=False
)
test_df.to_csv(
    test_path,
    index=False
)
log("\nSaved:")
log(train_path)
log(test_path)

log("\nSummary")
log("----------------")
log(f"Train samples: {len(train_df)}")
log(f"Test samples: {len(test_df)}")
log(f"Train positives: {train_df['EL'].sum()}")
log(f"Test positives: {test_df['EL'].sum()}")
log("Peptide overlap: 0")
log("\nSUCCESS")
PY


conda run -n gpmhc python save_dq_splits.py

Saving DQ-only train/test splits
Loaded dataframe: (1403497, 42)
DQ-only dataframe: (46317, 42)
Train: (26454, 42)
Test: (19863, 42)

Saved:
dq_dataset/DQ_train.csv
dq_dataset/DQ_test.csv

Summary
----------------
Train samples: 26454
Test samples: 19863
Train positives: 13668
Test positives: 2121
Peptide overlap: 0

SUCCESS



In [ ]:
%%bash
cd /content/drive/MyDrive/gpmhc/gpmhc_train

export MPLBACKEND=Agg

cat > dq_train_sanity.py <<'PY'

import json
import torch
import pandas as pd

from torch.utils.data import DataLoader
from sklearn.metrics import average_precision_score, roc_auc_score

print("Starting DQ pretrain sanity check", flush=True)

from gpmhc.learner import json_to_learner
from gpmhc.data import cleanup_schema, tokenize, dataset

with open("models/baseline_model/json_input.json") as f:
    config = json.load(f)

print("Config loaded", flush=True)

learner = json_to_learner(config)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

learner.model.to(device)
learner.model.eval()

print("Model initialized", flush=True)
print("Device:", device, flush=True)

train_df = pd.read_csv(
    "dq_dataset/DQ_train.csv",
    low_memory=False
)

test_df = pd.read_csv(
    "dq_dataset/DQ_test.csv",
    low_memory=False
)

print("Train:", train_df.shape, flush=True)
print("Test:", test_df.shape, flush=True)

assert set(train_df["split"]) == {"train"}
assert set(test_df["split"]) == {"test"}

assert set(train_df["peptide"]).isdisjoint(
    set(test_df["peptide"])
)

print("Leakage checks passed", flush=True)

schema_options = config["dataloader_options"]["csv_to_df"]["schema_options"]

train_df = cleanup_schema(
    train_df,
    schema_options
)

test_df = cleanup_schema(
    test_df,
    schema_options
)

print("Schema cleanup done", flush=True)


x_train = tokenize(
    train_df,
    learner.arch.tokenizer
)

y_train = train_df["EL"].values.astype(int)


x_test = tokenize(
    test_df,
    learner.arch.tokenizer
)

y_test = test_df["EL"].values.astype(int)


print("Train tokens:", x_train.shape, flush=True)
print("Test tokens:", x_test.shape, flush=True)

train_loader = DataLoader(
    dataset(x_train, y_train),
    batch_size=16,
    shuffle=True
)

test_loader = DataLoader(
    dataset(x_test, y_test),
    batch_size=16,
    shuffle=False
)


print("Dataloaders ready", flush=True)

with torch.no_grad():

    xb, yb = next(iter(train_loader))

    xb = xb.to(device)
    yb = yb.to(device)

    print("Batch:", xb.shape, flush=True)

    pred = learner.model(xb)

    print(
        "Prediction:",
        pred.shape,
        pred.device,
        flush=True
    )


print("SUCCESS: DQ pretrain pipeline loads correctly", flush=True)
PY

conda run -n gpmhc python dq_train_sanity.py

Starting DQ pretrain sanity check
Config loaded
Model initialized
Device: cuda
Train: (26454, 42)
Test: (19863, 42)
Leakage checks passed
Schema cleanup done
Train tokens: (26454, 728)
Test tokens: (19863, 728)
Dataloaders ready
Batch: torch.Size([16, 728])
Prediction: torch.Size([16, 32]) cuda:0
SUCCESS: DQ pretrain pipeline loads correctly



/usr/local/envs/gpmhc/lib/python3.10/site-packages/accelerate/utils/torch_xla.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/usr/local/envs/gpmhc/lib/python3.10/site-packages/dgl/heterograph.py:92: DGLWarning: Recommend creating graphs by `dgl.graph(data)` instead of `dgl.DGLGraph(data)`.
  dgl_warning(



In [ ]:
%%bash
cd /content/drive/MyDrive/gpmhc/gpmhc_train

export MPLBACKEND=Agg

cat > dq_finetune.py <<'PY'

import os
import json
import torch
import pandas as pd

from torch.utils.data import DataLoader
from sklearn.metrics import average_precision_score, roc_auc_score

from gpmhc.learner import json_to_learner
from gpmhc.data import cleanup_schema, tokenize, dataset


def log(x):
    print(x, flush=True)


log("Starting DQ fine-tuning")

EPOCHS = 20
BATCH_SIZE = 32
LR = 1e-5

SAVE_DIR = "models/dq_finetune"

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

with open(
    "models/baseline_model/json_input.json"
) as f:
    config = json.load(f)


learner = json_to_learner(config)


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


learner.model.to(device)

log(f"Device: {device}")


train_df = pd.read_csv(
    "dq_dataset/DQ_train.csv",
    low_memory=False
)

test_df = pd.read_csv(
    "dq_dataset/DQ_test.csv",
    low_memory=False
)


assert set(train_df["split"]) == {"train"}
assert set(test_df["split"]) == {"test"}

assert set(train_df["peptide"]).isdisjoint(
    set(test_df["peptide"])
)


log(f"Train: {train_df.shape}")
log(f"Test: {test_df.shape}")


schema_options = config[
    "dataloader_options"
]["csv_to_df"]["schema_options"]


train_df = cleanup_schema(
    train_df,
    schema_options
)

test_df = cleanup_schema(
    test_df,
    schema_options
)


# tokenize

x_train = tokenize(
    train_df,
    learner.arch.tokenizer
)

y_train = train_df["EL"].values.astype("float32")


x_test = tokenize(
    test_df,
    learner.arch.tokenizer
)

y_test = test_df["EL"].values.astype("float32")


train_loader = DataLoader(
    dataset(x_train, y_train),
    batch_size=BATCH_SIZE,
    shuffle=True
)


test_loader = DataLoader(
    dataset(x_test, y_test),
    batch_size=BATCH_SIZE,
    shuffle=False
)


log("Data ready")

optimizer = torch.optim.AdamW(
    learner.model.parameters(),
    lr=LR
)


loss_fn = torch.nn.BCEWithLogitsLoss()

learner.model.eval()

preds = []
labels = []

with torch.no_grad():

    for xb, yb in test_loader:

        xb = xb.to(device)

        logits = learner.model(xb)

        if logits.ndim > 1:
            logits = logits[:,12]

        probs = torch.sigmoid(logits)

        preds.extend(
            probs.cpu().numpy()
        )

        labels.extend(
            yb.numpy()
        )


init_ap = average_precision_score(
    labels,
    preds
)

init_auc = roc_auc_score(
    labels,
    preds
)

log(
    f"Initial model "
    f"AP={init_ap:.4f} "
    f"AUC={init_auc:.4f}"
)

for epoch in range(EPOCHS):

    learner.model.train()

    total_loss = 0


    for xb, yb in train_loader:

        xb = xb.to(device)
        yb = yb.to(device)


        optimizer.zero_grad()


        logits = learner.model(xb)


        # model outputs [batch,32]
        # collapse if needed
        # DQ allotypes map to slot 12 in this schema
        if logits.ndim > 1:
            logits = logits[:,12]


        loss = loss_fn(
            logits,
            yb
        )


        loss.backward()

        optimizer.step()


        total_loss += loss.item()



    avg_loss = total_loss / len(train_loader)

    if epoch == 0:
    best_ap = 0

    if ap > best_ap:
        best_ap = ap

        torch.save(
            {
                "epoch": epoch+1,
                "model_state_dict": learner.model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "AP": ap,
                "AUC": auc
            },
            f"{SAVE_DIR}/best_dq_model.pth"
        )

        log(f"New best model saved AP={ap:.4f}")

    learner.model.eval()

    preds = []
    labels = []


    with torch.no_grad():

        for xb, yb in test_loader:

            xb = xb.to(device)

            logits = learner.model(xb)


            if logits.ndim > 1:
                logits = logits[:,12] # DQ allotypes map to slot 12 in this schema


            probs = torch.sigmoid(logits)


            preds.extend(
                probs.cpu().numpy()
            )

            labels.extend(
                yb.numpy()
            )


    ap = average_precision_score(
        labels,
        preds
    )

    auc = roc_auc_score(
        labels,
        preds
    )


    log(
        f"Epoch {epoch+1}/{EPOCHS} "
        f"loss={avg_loss:.4f} "
        f"AP={ap:.4f} "
        f"AUC={auc:.4f}"
    )


    torch.save(
        {
            "epoch": epoch+1,
            "model_state_dict": learner.model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "AP": ap,
            "AUC": auc
        },
        f"{SAVE_DIR}/dq_epoch_{epoch+1}.pth"
    )


log("DQ fine-tuning complete")

PY


conda run -n gpmhc python dq_finetune.py

Starting DQ fine-tuning
Device: cuda
Train: (26454, 42)
Test: (19863, 42)
Data ready
Initial model AP=0.0725 AUC=0.3258
Epoch 1/20 loss=0.6586 AP=0.2625 AUC=0.7616
Epoch 2/20 loss=0.5519 AP=0.2714 AUC=0.7699
Epoch 3/20 loss=0.5216 AP=0.2886 AUC=0.7777
Epoch 4/20 loss=0.5053 AP=0.3029 AUC=0.7856
Epoch 5/20 loss=0.4926 AP=0.3169 AUC=0.7917
Epoch 6/20 loss=0.4857 AP=0.3303 AUC=0.7948
Epoch 7/20 loss=0.4797 AP=0.3401 AUC=0.8000
Epoch 8/20 loss=0.4750 AP=0.3508 AUC=0.8038
Epoch 9/20 loss=0.4695 AP=0.3572 AUC=0.8064
Epoch 10/20 loss=0.4671 AP=0.3655 AUC=0.8078
Epoch 11/20 loss=0.4587 AP=0.3723 AUC=0.8099
Epoch 12/20 loss=0.4556 AP=0.3830 AUC=0.8137
Epoch 13/20 loss=0.4537 AP=0.3909 AUC=0.8164
Epoch 14/20 loss=0.4500 AP=0.3937 AUC=0.8172
Epoch 15/20 loss=0.4457 AP=0.3988 AUC=0.8185
Epoch 16/20 loss=0.4450 AP=0.4038 AUC=0.8207
Epoch 17/20 loss=0.4404 AP=0.4076 AUC=0.8206
Epoch 18/20 loss=0.4367 AP=0.4130 AUC=0.8226
Epoch 19/20 loss=0.4322 AP=0.4129 AUC=0.8217
Epoch 20/20 loss=0.4312 AP=0.4237 

/usr/local/envs/gpmhc/lib/python3.10/site-packages/accelerate/utils/torch_xla.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/usr/local/envs/gpmhc/lib/python3.10/site-packages/dgl/heterograph.py:92: DGLWarning: Recommend creating graphs by `dgl.graph(data)` instead of `dgl.DGLGraph(data)`.
  dgl_warning(



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

%%bash

cd /content/drive/MyDrive/gpmhc/gpmhc_train

mkdir -p experiments
mkdir -p experiments/001_DQ_only/{scripts,logs,checkpoints,metrics,dataset}
mkdir -p experiments/002_HLAII_heterodimer/{scripts,logs,checkpoints,metrics,dataset}
mkdir -p experiments/003_HLAII_heterodimer_DQ_finetune/{scripts,logs,checkpoints,metrics,dataset}

echo "Experiment folders created"

Experiment folders created


In [ ]:
%%bash

cd /content/drive/MyDrive/gpmhc/gpmhc_train

cp dq_finetune.py \
experiments/001_DQ_only/scripts/

cp inspect_output.py \
experiments/001_DQ_only/scripts/

cp -r dq_dataset \
experiments/001_DQ_only/dataset/

echo "DQ experiment archived"

DQ experiment archived
